# Решения: дисбаланс и метрики

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('bank_marketing_slim.csv')
df = pd.read_csv(CSV_PATH)
target = (df['y'] == 'yes').astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [ ]:
y_true = target.to_numpy()
baseline_pred = np.zeros_like(y_true)
acc_base = float(accuracy_score(y_true, baseline_pred))
recall_base = float(recall_score(y_true, baseline_pred, zero_division=0))
feature_columns = [c for c in df.columns if c not in ('y', 'duration')]
X = pd.get_dummies(df[feature_columns], drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.25, random_state=62, stratify=target
)
model = LogisticRegression(max_iter=1200)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.50).astype(int)
cm = confusion_matrix(y_test, pred)
metrics = {
    'accuracy': float(accuracy_score(y_test, pred)),
    'precision': float(precision_score(y_test, pred, zero_division=0)),
    'recall': float(recall_score(y_test, pred, zero_division=0)),
    'f1': float(f1_score(y_test, pred, zero_division=0)),
}
rows = []
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    cur = (proba >= thr).astype(int)
    rows.append(
        {
            'threshold': thr,
            'precision': float(precision_score(y_test, cur, zero_division=0)),
            'recall': float(recall_score(y_test, cur, zero_division=0)),
            'f1': float(f1_score(y_test, cur, zero_division=0)),
        }
    )
table = pd.DataFrame(rows)
best_idx = int(table['f1'].idxmax())
best_thr = float(table.loc[best_idx, 'threshold'])
IMBALANCE_NOTE = (
    'На дисбалансе accuracy может выглядеть высокой даже у модели, которая почти всегда говорит no. '
    'Поэтому порог выбираем по precision/recall/F1 и проверяем матрицу ошибок.'
)
NOTE = (
    f'По F1 лучшим оказался порог {best_thr:.2f}; он даёт баланс между пропусками yes и ложными срабатываниями. '
    'Для бизнес-решения дополнительно смотрим recall как риск недозвона целевых клиентов.'
)
print('baseline acc=', acc_base, 'baseline recall=', recall_base)
print('cm=\n', cm)
print(metrics)
print(table)
print(NOTE)